In [ ]:
import numpy as np
import matplotlib.pyplot as plt

lista_uni = None
indice_uni = 0

#Configuracion de distribucion uniforme: 

def configurar_uniformes():
    global lista_uni, indice_uni
    print(" FUENTE DE UNIFORMES")
    resp = input("Usar lista personalizada? (s/n): ").lower()
    if resp == 's':
        entrada = input("Numeros en (0,1) separados por comas ej:0.1,0.5,0.9: ")
        try:
            valores = [float(x.strip()) for x in entrada.split(',')]
            if any(v <= 0 or v >= 1 for v in valores):
                print("Error: todos deben estar en (0,1). Usando aleatorios.")
                lista_uni = None
            else:
                lista_uni = valores
                indice_uni = 0
                print(f"Lista cargada con {len(valores)} valores. Reutilizacion ciclica.")
        except:
            print("Formato invalido. Usando aleatorios.")
            lista_uni = None
    else:
        lista_uni = None
        print("Usando uniformes aleatorios (numpy).")

def obtener_uniforme():
    global indice_uni, lista_uni
    if lista_uni is not None:
        if len(lista_uni) == 0:
            return np.random.uniform(0,1)
        u = lista_uni[indice_uni]
        indice_uni = (indice_uni + 1) % len(lista_uni)
        return u
    else:
        return np.random.uniform(0,1)

#Generadores de transformada inversa: 

def generar_weibull(alfa, n=1):
    u = np.array([obtener_uniforme() for _ in range(n)])
    return (-np.log(u))**(1/alfa)

def generar_gumbel(n=1):
    u = np.array([obtener_uniforme() for _ in range(n)])
    return -np.log(-np.log(u))

def generar_cauchy(n=1):
    u = np.array([obtener_uniforme() for _ in range(n)])
    return np.tan(np.pi*(u - 0.5))

def generar_logistica(n=1):
    u = np.array([obtener_uniforme() for _ in range(n)])
    return np.log(u/(1-u))

def generar_pareto(alfa, n=1):
    u = np.array([obtener_uniforme() for _ in range(n)])
    return u**(-1/alfa)

#Densidades y CDF teoricas:

def pdf_weibull(x, alfa):
    # f(x) = alfa * x^(alfa-1) * exp(-x^alfa)
    return alfa * (x**(alfa-1)) * np.exp(-x**alfa)

def cdf_weibull(x, alfa):
    return 1 - np.exp(-x**alfa)

def pdf_gumbel(x):
    # f(x) = e^(-x - e^{-x})
    return np.exp(-x - np.exp(-x))

def cdf_gumbel(x):
    return np.exp(-np.exp(-x))

def pdf_cauchy(x):
    return 1/(np.pi*(1 + x**2))

def cdf_cauchy(x):
    return 0.5 + np.arctan(x)/np.pi

def pdf_logistica(x):
    e = np.exp(-x)
    return e/(1+e)**2

def cdf_logistica(x):
    return 1/(1+np.exp(-x))

def pdf_pareto(x, alfa):
    # soporte x>=1
    return alfa * x**(-alfa-1)

def cdf_pareto(x, alfa):
    return 1 - x**(-alfa)

#Estadisticas y gráficas

def mostrar_estadisticos(muestra, media_teo, var_teo, nombre):
    print(f"\n=== {nombre} ===")
    print(f"Media muestral: {np.mean(muestra):.6f}")
    if media_teo is not None:
        print(f"Media teorica : {media_teo:.6f}")
    else:
        print("Media teorica : no definida")
    print(f"Varianza muestral: {np.var(muestra, ddof=1):.6f}")
    if var_teo is not None:
        print(f"Varianza teorica : {var_teo:.6f}")
    else:
        print("Varianza teorica : no definida")
    print(f"Desv estandar muestral: {np.std(muestra, ddof=1):.6f}")
    if var_teo is not None:
        print(f"Desv teorica        : {np.sqrt(var_teo):.6f}")
    else:
        print("Desv teorica        : no definida")

def graficar(muestra, nombre, pdf_func, cdf_func, rango_x=None, alfa=None):
    n = len(muestra)
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12,4))
    fig.suptitle(f'Distribucion {nombre}', fontsize=14)

    # Recorte de colas para visualización
    if rango_x is None:
        x_min, x_max = np.min(muestra), np.max(muestra)
        if nombre == "Cauchy":
            x_min, x_max = -10, 10
            muestra_plot = muestra[(muestra > -10) & (muestra < 10)]
        elif nombre == "Pareto":
            x_max = min(10, np.max(muestra))
            muestra_plot = muestra[muestra <= x_max]
        else:
            muestra_plot = muestra
    else:
        x_min, x_max = rango_x
        muestra_plot = muestra

    # Histograma vs densidad teórica
    ax1.hist(muestra_plot, bins=50, density=True, alpha=0.6, label='Muestra')
    x_vals = np.linspace(x_min, x_max, 200)
    if alfa is not None and nombre in ["Weibull","Pareto"]:
        y_vals = pdf_func(x_vals, alfa)
    else:
        y_vals = pdf_func(x_vals)
    ax1.plot(x_vals, y_vals, 'r-', label='Densidad teorica')
    ax1.set_xlabel('x')
    ax1.set_ylabel('Densidad')
    ax1.legend()
    ax1.set_title('Histograma vs f(x) teorica')

    # CDF empírica vs teórica
    x_ordenado = np.sort(muestra)
    ecdf = np.arange(1, n+1)/n
    ax2.plot(x_ordenado, ecdf, 'b.', markersize=2, label='CDF empirica')
    x_cdf = np.linspace(np.min(muestra), np.max(muestra), 200)
    if alfa is not None and nombre in ["Weibull","Pareto"]:
        y_cdf = cdf_func(x_cdf, alfa)
    else:
        y_cdf = cdf_func(x_cdf)
    ax2.plot(x_cdf, y_cdf, 'r-', label='CDF teorica')
    ax2.set_xlabel('x')
    ax2.set_ylabel('F(x)')
    ax2.legend()
    ax2.set_title('Funcion de distribucion')

    plt.tight_layout()
    plt.show()

def menu():
    print("    GENERADOR TRANSFORMADA INVERSA")
    print("1. Weibull We(a1)")
    print("2. Gumbel Gu(01)")
    print("3. Cauchy C(01)")
    print("4. Logistica L(01)")
    print("5. Pareto Par(a)")
    print("6. Cambiar fuente de uniformes")
    print("0. Salir")
    return input("Seleccione opcion: ")

print("Bienvenido. Configure la fuente de uniformes:")
configurar_uniformes()

while True:
    op = menu()
    if op == '0':
        print("Programa finalizado.")
        break
    elif op == '6':
        configurar_uniformes()
        continue

    try:
        n = int(input("Numero de valores a generar: "))
        if n <= 0:
            print("Error: n debe ser >0.")
            continue

        if op == '1':   # Weibull
            alfa = float(input("Ingrese alfa>0: "))
            if alfa <= 0:
                print("Error: alfa>0.")
                continue
            muestra = generar_weibull(alfa, n)
            print(f"Media muestral: {np.mean(muestra):.6f}")
            print(f"Varianza muestral: {np.var(muestra, ddof=1):.6f}")
            print(f"Desv estandar muestral: {np.std(muestra, ddof=1):.6f}")
            graficar(muestra, "Weibull", pdf_weibull, cdf_weibull, alfa=alfa)

        elif op == '2':   # Gumbel
            muestra = generar_gumbel(n)
            media_teo = 0.5772156649  # Euler-Mascheroni
            var_teo = np.pi**2 / 6
            mostrar_estadisticos(muestra, media_teo, var_teo, "Gumbel")
            graficar(muestra, "Gumbel", pdf_gumbel, cdf_gumbel)

        elif op == '3':   # Cauchy
            muestra = generar_cauchy(n)
            mostrar_estadisticos(muestra, None, None, "Cauchy")
            graficar(muestra, "Cauchy", pdf_cauchy, cdf_cauchy, rango_x=(-10,10))

        elif op == '4':   # Logistica
            muestra = generar_logistica(n)
            media_teo = 0.0
            var_teo = np.pi**2 / 3
            mostrar_estadisticos(muestra, media_teo, var_teo, "Logistica")
            graficar(muestra, "Logistica", pdf_logistica, cdf_logistica)

        elif op == '5':   # Pareto
            alfa = float(input("Ingrese alfa>0: "))
            if alfa <= 0:
                print("Error: alfa>0.")
                continue
            muestra = generar_pareto(alfa, n)
            if alfa > 1:
                media_teo = alfa/(alfa-1)
            else:
                media_teo = None
            if alfa > 2:
                var_teo = alfa/((alfa-1)**2*(alfa-2))
            else:
                var_teo = None
            mostrar_estadisticos(muestra, media_teo, var_teo, "Pareto")
            graficar(muestra, "Pareto", pdf_pareto, cdf_pareto, rango_x=(1, min(10, np.max(muestra))), alfa=alfa)

        else:
            print("Opcion no valida.")

    except ValueError:
        print("Error: ingrese un valor numerico.")
    except Exception as e:
        print(f"Error inesperado: {e}")

Bienvenido. Configure la fuente de uniformes:
 FUENTE DE UNIFORMES
